# On-Policy Policy Gradients, Head-to-Head: REINFORCE → A2C → PPO

**Goal.** Compare three *on-policy policy-gradient algorithms* on the same task,
with the **same policy/value networks, the same environment, and the same random
seeds** — so that *the only thing that changes is the algorithm itself*.

| Algorithm | Advantage estimate | Updates per batch | Key idea |
|---|---|---|---|
| **REINFORCE** | Monte-Carlo return-to-go − V(s) | **1** | the classic policy gradient + a value baseline |
| **A2C** (a.k.a. VPG) | **GAE(λ)** + value critic | **1** | bootstrapped critic lowers variance |
| **PPO** | **GAE(λ)** + value critic | **many** (epochs × minibatches) | clipped surrogate lets you safely reuse each batch |

**The story the plots should tell:** REINFORCE is *high-variance*; A2C adds a
*critic* to tame that variance; PPO adds a *clipped objective + multi-epoch
updates* for the best **stability** and **sample efficiency**.

> 🕒 This runs in a **few minutes on a free Colab CPU** — small nets, small env,
> 3 seeds each. This is a sibling of the "reward/weight designs within REINFORCE"
> notebook; here we vary the *algorithm*, not the reward.


In [ ]:
%pip install -q "gymnasium>=0.29" "pygame>=2.5"


## Global configuration

Everything you might want to tweak lives here. The defaults are tuned so each
algorithm visibly learns CartPole in a couple of minutes on CPU.

In [ ]:
import time, math, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Categorical
import gymnasium as gym
import matplotlib
import matplotlib.pyplot as plt

# ----------------------------------------------------------------------------
# GLOBAL CONFIG — tweak these knobs
# ----------------------------------------------------------------------------
ENV_ID      = "CartPole-v1"   # classic control benchmark, "solved" at 475/500
GAMMA       = 0.99            # discount factor
LAM         = 0.95           # GAE(lambda) trace-decay (used by A2C & PPO)
ITERS       = 60             # number of on-policy training iterations (batches)
BATCH       = 2000           # env steps collected per iteration
SEEDS       = [0, 1, 2]      # run every algorithm under all of these seeds
HIDDEN      = 64             # hidden width of the (small) MLPs

# Learning rates (separate for policy & value heads)
LR_POLICY   = 3e-3
LR_VALUE    = 1e-2

# PPO-specific knobs
PPO_EPOCHS  = 10             # how many times PPO reuses each collected batch
CLIP_EPS    = 0.2            # PPO clipping range  (1-eps, 1+eps)
MINIBATCHES = 4             # minibatches per epoch for PPO
ENT_COEF    = 0.0           # entropy bonus (0 keeps the comparison clean)

SOLVED      = 475.0         # CartPole-v1 "solved" threshold (avg return)
DEVICE      = torch.device("cpu")   # free Colab: CPU is plenty for this

print("torch :", torch.__version__)
print("gym   :", gym.__version__)
print("device:", DEVICE)


In [ ]:
def set_seed(seed: int):
    """Seed python / numpy / torch so runs are reproducible."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


## The task: CartPole-v1

A pole is balanced on a cart; push the cart **left** or **right** each step.
You get **+1 reward per timestep** the pole stays up, for up to 500 steps.
Observation is 4 numbers (cart position/velocity, pole angle/angular velocity);
two discrete actions. Below is a **random agent** — it falls over almost
immediately.

In [ ]:
from matplotlib import animation
from IPython.display import HTML

def render_episode_frames(policy=None, seed=0, max_steps=500):
    """Roll out one episode in rgb_array mode and return a list of frames.
    If policy is None, acts randomly."""
    env = gym.make(ENV_ID, render_mode="rgb_array")
    obs, _ = env.reset(seed=seed)
    frames = []
    for _ in range(max_steps):
        frames.append(env.render())
        if policy is None:
            action = env.action_space.sample()
        else:
            with torch.no_grad():
                logits = policy(torch.as_tensor(obs, dtype=torch.float32).unsqueeze(0))
                action = int(torch.argmax(logits, dim=-1).item())  # greedy
        obs, _, term, trunc, _ = env.step(action)
        if term or trunc:
            break
    env.close()
    return frames

def frames_to_html(frames, interval=40):
    """Turn a list of rgb frames into an inline HTML5 animation (Colab-friendly)."""
    plt.ioff()
    fig = plt.figure(figsize=(4, 3))
    plt.axis("off")
    im = plt.imshow(frames[0])
    def update(i):
        im.set_data(frames[i])
        return (im,)
    anim = animation.FuncAnimation(fig, update, frames=len(frames),
                                   interval=interval, blit=True)
    html = HTML(anim.to_jshtml())
    plt.close(fig)
    plt.ion()
    return html

# Random agent — usually lasts only a handful of steps.
_rand_frames = render_episode_frames(policy=None, seed=0)
print(f"Random agent survived {len(_rand_frames)} steps")
frames_to_html(_rand_frames)


## Networks

Both heads are tiny MLPs and are **identical across all three algorithms**.
The policy outputs logits for a `Categorical` distribution over the 2 actions;
the value net outputs a scalar baseline V(s).

In [ ]:
def mlp(in_dim, out_dim, hidden=HIDDEN):
    """A small 2-hidden-layer MLP with tanh activations."""
    return nn.Sequential(
        nn.Linear(in_dim, hidden), nn.Tanh(),
        nn.Linear(hidden, hidden), nn.Tanh(),
        nn.Linear(hidden, out_dim),
    )

class PolicyNet(nn.Module):
    """Maps an observation to action logits (Categorical)."""
    def __init__(self, obs_dim, n_actions):
        super().__init__()
        self.net = mlp(obs_dim, n_actions)
    def forward(self, obs):
        return self.net(obs)              # logits
    def dist(self, obs):
        return Categorical(logits=self.forward(obs))

class ValueNet(nn.Module):
    """Maps an observation to a scalar state-value baseline V(s)."""
    def __init__(self, obs_dim):
        super().__init__()
        self.net = mlp(obs_dim, 1)
    def forward(self, obs):
        return self.net(obs).squeeze(-1)  # shape (N,)


## Rollout collection (on-policy)

Each iteration we collect ~`BATCH` environment steps with the **current**
policy, recording observations, actions, rewards, log-probs, value estimates,
and episode boundaries. On-policy means: after we update, we throw this data
away and collect fresh data (PPO is the exception — it *reuses* its batch for
several epochs, but still discards it before the next iteration).

In [ ]:
def collect_rollout(env, policy, value, batch_steps, seed_iter):
    """Run the current policy for ~batch_steps env steps.

    Returns a dict of numpy/torch arrays plus the list of completed-episode
    returns (for the learning curve)."""
    obs_buf, act_buf, rew_buf = [], [], []
    logp_buf, val_buf = [], []
    done_buf = []            # 1.0 if this step *terminated* an episode (real terminal)
    ep_returns = []          # sum of rewards for each finished episode

    obs, _ = env.reset(seed=seed_iter)
    ep_ret = 0.0
    steps = 0
    while steps < batch_steps:
        obs_t = torch.as_tensor(obs, dtype=torch.float32).unsqueeze(0)
        with torch.no_grad():
            dist = policy.dist(obs_t)
            action = dist.sample()
            logp = dist.log_prob(action)
            val = value(obs_t)

        a = int(action.item())
        next_obs, rew, term, trunc, _ = env.step(a)

        obs_buf.append(np.asarray(obs, dtype=np.float32))
        act_buf.append(a)
        rew_buf.append(float(rew))
        logp_buf.append(float(logp.item()))
        val_buf.append(float(val.item()))
        # Only a *true* termination (term) zeroes the bootstrap; truncation does not.
        done_buf.append(1.0 if term else 0.0)

        ep_ret += float(rew)
        obs = next_obs
        steps += 1

        if term or trunc:
            ep_returns.append(ep_ret)
            ep_ret = 0.0
            obs, _ = env.reset()

    # Bootstrap value for the final, possibly-unfinished state.
    with torch.no_grad():
        last_val = float(value(torch.as_tensor(obs, dtype=torch.float32).unsqueeze(0)).item())

    return {
        "obs":  np.asarray(obs_buf, dtype=np.float32),
        "act":  np.asarray(act_buf, dtype=np.int64),
        "rew":  np.asarray(rew_buf, dtype=np.float32),
        "logp": np.asarray(logp_buf, dtype=np.float32),
        "val":  np.asarray(val_buf, dtype=np.float32),
        "done": np.asarray(done_buf, dtype=np.float32),
        "last_val": last_val,
        "ep_returns": ep_returns,
    }, ep_returns


## GAE(λ) and reward-to-go (numpy helpers)

**Generalized Advantage Estimation** trades off bias and variance with `λ`:
`A_t = δ_t + (γλ) δ_{t+1} + (γλ)^2 δ_{t+2} + …`, where the TD residual is
`δ_t = r_t + γ V(s_{t+1}) − V(s_t)`. The value target is `A_t + V(s_t)`.

REINFORCE instead uses the **Monte-Carlo return-to-go** `G_t = Σ γ^{k} r_{t+k}`
and a *non-bootstrapped* advantage `G_t − V(s_t)`.

In [ ]:
def compute_gae(rew, val, done, last_val, gamma=GAMMA, lam=LAM):
    """Generalized Advantage Estimation over a flat batch.

    `done[t]==1` marks a true terminal step (no bootstrap across it).
    `last_val` bootstraps the final, possibly-truncated state.
    Returns (advantages, value_targets), both shape (T,)."""
    T = len(rew)
    adv = np.zeros(T, dtype=np.float32)
    last_gae = 0.0
    for t in reversed(range(T)):
        nonterminal = 1.0 - done[t]
        next_val = last_val if t == T - 1 else val[t + 1]
        delta = rew[t] + gamma * next_val * nonterminal - val[t]
        last_gae = delta + gamma * lam * nonterminal * last_gae
        adv[t] = last_gae
    returns = adv + val          # value-function targets
    return adv, returns

def compute_reward_to_go(rew, done, gamma=GAMMA):
    """Discounted Monte-Carlo return-to-go G_t (resets at terminals).
    Used by REINFORCE. Returns shape (T,)."""
    T = len(rew)
    rtg = np.zeros(T, dtype=np.float32)
    running = 0.0
    for t in reversed(range(T)):
        if done[t] > 0.5:
            running = 0.0        # episode boundary -> fresh return
        running = rew[t] + gamma * running
        rtg[t] = running
    return rtg

def standardize(x, eps=1e-8):
    """Zero-mean, unit-std — stabilizes the policy gradient."""
    return (x - x.mean()) / (x.std() + eps)


## One `train(algo, seed)` to rule them all

A single training loop. The **only** thing that branches on `algo` is the
**update rule**:

* **REINFORCE** — advantage = `reward_to_go − V(s)` (MC, no bootstrap); **1**
  gradient step on the policy; fit the value baseline by regression.
* **A2C** — advantage = `GAE(λ)`; **1** gradient step on policy + value.
* **PPO** — advantage = `GAE(λ)`; **`PPO_EPOCHS`** passes over the batch in
  `MINIBATCHES` minibatches, each maximizing the **clipped surrogate**.

In every case we **standardize advantages** before the policy update.

In [ ]:
def train(algo, seed, iters=ITERS, batch=BATCH, verbose=False):
    """Train one algorithm under one seed. Returns (curve, policy).

    curve[i] = mean completed-episode return during iteration i."""
    assert algo in ("reinforce", "a2c", "ppo")
    set_seed(seed)
    env = gym.make(ENV_ID)
    # Seed the action space too so sampling is reproducible.
    try:
        env.action_space.seed(seed)
    except Exception:
        pass

    obs_dim = env.observation_space.shape[0]
    n_act   = env.action_space.n
    policy = PolicyNet(obs_dim, n_act).to(DEVICE)
    value  = ValueNet(obs_dim).to(DEVICE)
    opt_pi = torch.optim.Adam(policy.parameters(), lr=LR_POLICY)
    opt_v  = torch.optim.Adam(value.parameters(),  lr=LR_VALUE)

    curve = []
    for it in range(iters):
        # ---- 1) collect an on-policy batch -------------------------------
        data, ep_returns = collect_rollout(env, policy, value, batch, seed + it)
        mean_ret = float(np.mean(ep_returns)) if ep_returns else 0.0
        curve.append(mean_ret)

        obs_t  = torch.as_tensor(data["obs"],  dtype=torch.float32, device=DEVICE)
        act_t  = torch.as_tensor(data["act"],  dtype=torch.int64,   device=DEVICE)
        oldlogp = torch.as_tensor(data["logp"], dtype=torch.float32, device=DEVICE)

        # ---- 2) advantages & value targets (algorithm-specific) ----------
        if algo == "reinforce":
            rtg = compute_reward_to_go(data["rew"], data["done"], GAMMA)
            ret_t = torch.as_tensor(rtg, dtype=torch.float32, device=DEVICE)
            adv_np = rtg - data["val"]            # MC advantage with value baseline
        else:  # a2c & ppo both use GAE(lambda)
            adv_np, ret = compute_gae(data["rew"], data["val"], data["done"],
                                      data["last_val"], GAMMA, LAM)
            ret_t = torch.as_tensor(ret, dtype=torch.float32, device=DEVICE)
        adv_t = torch.as_tensor(standardize(adv_np), dtype=torch.float32, device=DEVICE)

        # ---- 3) update ---------------------------------------------------
        if algo in ("reinforce", "a2c"):
            # ONE gradient step on the policy ...
            dist = policy.dist(obs_t)
            logp = dist.log_prob(act_t)
            pi_loss = -(logp * adv_t).mean() - ENT_COEF * dist.entropy().mean()
            opt_pi.zero_grad(); pi_loss.backward(); opt_pi.step()
            # ... and ONE (well, a few regression steps) on the value baseline.
            v_iters = 1 if algo == "reinforce" else 1
            for _ in range(80):   # fit baseline well so it actually reduces variance
                v_loss = F.mse_loss(value(obs_t), ret_t)
                opt_v.zero_grad(); v_loss.backward(); opt_v.step()

        elif algo == "ppo":
            # MULTIPLE epochs, each split into minibatches, reusing this batch.
            N = obs_t.shape[0]
            mb_size = max(1, N // MINIBATCHES)
            idx = np.arange(N)
            for _ in range(PPO_EPOCHS):
                np.random.shuffle(idx)
                for start in range(0, N, mb_size):
                    mb = idx[start:start + mb_size]
                    mb_t = torch.as_tensor(mb, dtype=torch.int64, device=DEVICE)
                    dist = policy.dist(obs_t[mb_t])
                    logp = dist.log_prob(act_t[mb_t])
                    ratio = torch.exp(logp - oldlogp[mb_t])          # pi / pi_old
                    a = adv_t[mb_t]
                    unclipped = ratio * a
                    clipped = torch.clamp(ratio, 1 - CLIP_EPS, 1 + CLIP_EPS) * a
                    pi_loss = -torch.min(unclipped, clipped).mean() \
                              - ENT_COEF * dist.entropy().mean()
                    v_loss = F.mse_loss(value(obs_t[mb_t]), ret_t[mb_t])
                    opt_pi.zero_grad(); pi_loss.backward(); opt_pi.step()
                    opt_v.zero_grad(); v_loss.backward(); opt_v.step()

        if verbose and (it % 10 == 0 or it == iters - 1):
            print(f"  [{algo:9s} seed={seed}] iter {it:3d}  mean_return={mean_ret:6.1f}")

    env.close()
    return np.asarray(curve, dtype=np.float32), policy


## Run all three algorithms × all seeds

Same nets, same env, same seeds — only `algo` differs. We record each learning
curve and time the whole sweep.

In [ ]:
ALGOS = ["reinforce", "a2c", "ppo"]
results = {a: [] for a in ALGOS}     # results[algo] -> list of curves (one per seed)
trained_policies = {}                # keep one trained policy per algo (last seed)

t0 = time.time()
for algo in ALGOS:
    for seed in SEEDS:
        curve, pol = train(algo, seed, verbose=True)
        results[algo].append(curve)
        trained_policies[algo] = pol      # remember last seed's policy
    print(f"== {algo.upper():9s} done ==")
elapsed = time.time() - t0
print(f"\nTotal training time: {elapsed:.1f} s "
      f"({len(ALGOS)*len(SEEDS)} runs of {ITERS} iters x {BATCH} steps)")

# Stack into arrays of shape (n_seeds, ITERS)
curves = {a: np.stack(results[a], axis=0) for a in ALGOS}
for a in ALGOS:
    print(f"{a:9s} final mean return (avg over seeds): "
          f"{curves[a][:, -1].mean():.1f}")


## Plot 1 — Learning curves (mean ± std over seeds)

The shaded band is ±1 std across the 3 seeds. Watch how **REINFORCE** wobbles
(high variance), **A2C** is steadier thanks to its critic, and **PPO** climbs
fastest and holds near the **solved=475** line.

In [ ]:
plt.figure(figsize=(8, 5))
colors = {"reinforce": "tab:red", "a2c": "tab:orange", "ppo": "tab:green"}
labels = {"reinforce": "REINFORCE (MC baseline)",
          "a2c": "A2C (GAE, 1 step)",
          "ppo": f"PPO (GAE, {PPO_EPOCHS} epochs)"}
x = np.arange(1, ITERS + 1)
for a in ALGOS:
    mean = curves[a].mean(axis=0)
    std  = curves[a].std(axis=0)
    plt.plot(x, mean, color=colors[a], label=labels[a], linewidth=2)
    plt.fill_between(x, mean - std, mean + std, color=colors[a], alpha=0.18)
plt.axhline(SOLVED, color="gray", linestyle="--", linewidth=1, label=f"solved = {SOLVED:.0f}")
plt.xlabel("Iteration (on-policy batch)")
plt.ylabel("Mean episode return")
plt.title("REINFORCE vs A2C vs PPO on CartPole-v1")
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## Plot 2 — Sample efficiency & stability

**Left:** final performance (mean over seeds, last 5 iterations) as a bar chart
with per-seed error bars. **Right:** return vs **total environment steps** — the
fair axis for *sample efficiency*, since all algos see `BATCH` steps per
iteration. Note PPO does *more learning per environment step* because it reuses
each batch **`PPO_EPOCHS`** times (it pays in compute, not in samples).

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

# --- Left: final-performance bar chart (avg of last 5 iters) ---
finals_mean, finals_std = [], []
for a in ALGOS:
    last5 = curves[a][:, -5:].mean(axis=1)     # per-seed final perf
    finals_mean.append(last5.mean())
    finals_std.append(last5.std())
ax1.bar(range(len(ALGOS)), finals_mean, yerr=finals_std, capsize=6,
        color=[colors[a] for a in ALGOS])
ax1.set_xticks(range(len(ALGOS)))
ax1.set_xticklabels([a.upper() for a in ALGOS])
ax1.axhline(SOLVED, color="gray", linestyle="--", linewidth=1)
ax1.set_ylabel("Final mean return (last 5 iters)")
ax1.set_title("Final performance (mean ± std over seeds)")
ax1.grid(alpha=0.3, axis="y")

# --- Right: return vs total environment steps ---
steps_axis = np.arange(1, ITERS + 1) * BATCH
for a in ALGOS:
    mean = curves[a].mean(axis=0)
    ax2.plot(steps_axis, mean, color=colors[a], label=labels[a], linewidth=2)
ax2.axhline(SOLVED, color="gray", linestyle="--", linewidth=1, label=f"solved = {SOLVED:.0f}")
ax2.set_xlabel("Total environment steps")
ax2.set_ylabel("Mean episode return")
ax2.set_title("Sample efficiency (return vs env steps)")
ax2.legend(loc="lower right", fontsize=9)
ax2.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"PPO reuses each {BATCH}-step batch {PPO_EPOCHS} times "
      f"({PPO_EPOCHS}x{MINIBATCHES} = {PPO_EPOCHS*MINIBATCHES} gradient steps/iter), "
      f"while REINFORCE & A2C take just 1 policy step/iter.")


## Watch the trained PPO agent

Greedy rollout of the PPO policy from the last seed. Compare with the flailing
random agent at the top.

In [ ]:
_ppo_frames = render_episode_frames(policy=trained_policies["ppo"], seed=123)
print(f"Trained PPO agent survived {len(_ppo_frames)} steps "
      f"(max is 500 for CartPole-v1)")
frames_to_html(_ppo_frames)


## Takeaways & things to try

**The progression.**
1. **REINFORCE** is the purest policy gradient: `∇ E[Σ G_t ∇log π(a|s)]`, here
   with a learned value **baseline** to subtract a state-dependent constant.
   Unbiased, but the Monte-Carlo returns make it **high-variance** → noisy curves.
2. **A2C / VPG** swaps the MC return for a **bootstrapped critic via GAE(λ)**.
   `λ` interpolates between low-variance/high-bias TD (`λ→0`) and
   high-variance/low-bias MC (`λ→1`). Same single update, much smoother.
3. **PPO** keeps GAE but changes the *objective*: the **clipped surrogate**
   `min(r·A, clip(r, 1−ε, 1+ε)·A)` caps how far the new policy can move from the
   old one in a single batch. Because the step is *trust-region-like and safe*,
   PPO can take **many epochs / minibatches over the same data** — big gains in
   **sample efficiency and stability** for a modest compute cost.

**Why clip + multi-epoch helps.** Reusing on-policy data normally diverges:
after one step the data is off-policy and the gradient is wrong. The importance
ratio `r = π/π_old` corrects for this, and the clip removes the incentive to push
`r` far from 1 — so several gradient steps stay trustworthy instead of blowing up.

**Things to try.**
- **Turn off the critic** in A2C (set `compute_gae` baseline `val` to zeros, or
  use `reward_to_go` without the baseline) and watch variance explode.
- **Vary `CLIP_EPS`** (e.g. 0.05 vs 0.4): too small = slow, too large = PPO
  starts to behave like un-clipped multi-epoch A2C and can destabilize.
- **Vary `PPO_EPOCHS`** (1 → 20): 1 epoch ≈ A2C; too many overfits the batch.
- **Vary `LAM`** (0.0, 0.9, 1.0) to feel the GAE bias/variance trade-off.
- Add an **entropy bonus** (`ENT_COEF > 0`) to encourage exploration.

**Further reading (Notion).**
- [Vanilla Policy Gradient (VPG / A2C)](https://app.notion.com/p/37f95008d766819c87d7f4edad4034c9)
- [TRPO — the trust-region precursor to PPO](https://app.notion.com/p/37f95008d766817ba62ac91acb866037)
- [PPO — clipped surrogate objective](https://app.notion.com/p/37f95008d76681ae994febc1f7e89aa5)
